# PredictIQ EDA

Exploratory checks for the Microsoft Azure predictive maintenance dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_DIR = Path("..") / "data" / "raw"

In [ ]:
csv_paths = sorted(DATA_DIR.glob("*.csv"))
assert len(csv_paths) == 5, f"Expected 5 CSVs, found {len(csv_paths)}: {[p.name for p in csv_paths]}"

data = {path.stem.lower(): pd.read_csv(path) for path in csv_paths}
data.keys()

In [ ]:
for name, df in data.items():
    print(f"\n{name}")
    print("shape:", df.shape)
    print("dtypes:")
    print(df.dtypes)
    display(df.head(3))

In [ ]:
telemetry = data["pdms_telemetry"]
errors = data["pdms_errors"]
failures = data["pdms_failures"]
maint = data["pdms_maint"]
machines = data["pdms_machines"]

for df in [telemetry, errors, failures, maint]:
    df["datetime"] = pd.to_datetime(df["datetime"])

for name, df in {
    "telemetry": telemetry,
    "errors": errors,
    "failures": failures,
    "maint": maint,
}.items():
    print(f"{name}: {df['datetime'].min()} to {df['datetime'].max()}")

In [ ]:
failure_counts = failures["failure"].value_counts().sort_index()
ax = failure_counts.plot(kind="bar", figsize=(7, 4), color="#3b82f6")
ax.set_title("Failure counts per component")
ax.set_xlabel("Component")
ax.set_ylabel("Failures")
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
error_counts = errors["errorID"].value_counts().sort_index()
ax = error_counts.plot(kind="bar", figsize=(7, 4), color="#ef4444")
ax.set_title("Error frequency per errorID")
ax.set_xlabel("errorID")
ax.set_ylabel("Frequency")
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
telemetry_cols = ["volt", "rotate", "pressure", "vibration"]
plt.figure(figsize=(6, 5))
sns.heatmap(telemetry[telemetry_cols].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Telemetry correlation heatmap")
plt.tight_layout()

In [ ]:
failures_per_machine = failures.groupby("machineID").size()
all_machine_ids = pd.Index(machines["machineID"].unique(), name="machineID")
failure_bucket_counts = failures_per_machine.reindex(all_machine_ids, fill_value=0).map(
    lambda count: "0" if count == 0 else "1" if count == 1 else "2+"
).value_counts().reindex(["0", "1", "2+"], fill_value=0)

print("Machines by failure count bucket:")
print(failure_bucket_counts)

In [ ]:
error_events = errors[["machineID", "datetime", "errorID"]].rename(columns={"datetime": "error_datetime"})
failure_events = failures[["machineID", "datetime", "failure"]].rename(columns={"datetime": "failure_datetime"})
error_events = error_events.sort_values(["machineID", "error_datetime"])
failure_events = failure_events.sort_values(["machineID", "failure_datetime"])

next_failures = pd.merge_asof(
    error_events,
    failure_events,
    left_on="error_datetime",
    right_on="failure_datetime",
    by="machineID",
    direction="forward",
)
next_failures = next_failures.dropna(subset=["failure"])
next_failures["lag_hours"] = (
    next_failures["failure_datetime"] - next_failures["error_datetime"]
).dt.total_seconds() / 3600

avg_lag_by_machine = next_failures.groupby("machineID")["lag_hours"].mean()
print("Average error-to-next-failure lag in hours per machine:")
display(avg_lag_by_machine.to_frame())
print("Overall average lag hours:", avg_lag_by_machine.mean())